# Milestone 3 dan Final: EDA dan Insights

Analisis eksplorasi **MedFact-Synth** dari data hasil cleaning (`data/processed/medfact_clean.parquet`) dengan 7 visualisasi interaktif Plotly dan 6 insight utama.

## Keterbatasan Dataset

Dataset tidak memiliki kolom tanggal maupun lokasi geografis, skemanya hanya teks klaim medis dan metadata PubMed. Karena itu:

- **Analisis spasial tidak dilakukan**: tidak ada data lokasi sama sekali.
- **Analisis temporal memakai proxy**: PubMed ID (PMID) bertambah seiring waktu artikel masuk ke PubMed, sehingga PMID yang lebih besar *cenderung* lebih baru. Ini **bukan tanggal asli**, hanya urutan kasar, dan diberi label proxy pada hasilnya.

In [1]:
from pathlib import Path

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
DATA_GLOB = (RAW_DIR / 'data' / '*.parquet').as_posix()

import plotly.express as px
import polars as pl

CLEAN_PATH = PROJECT_ROOT / 'data' / 'processed' / 'medfact_clean.parquet'

if not CLEAN_PATH.exists():
    raise FileNotFoundError(f'{CLEAN_PATH} belum ada. Jalankan 02_data_cleaning.ipynb terlebih dahulu.')

lf = pl.scan_parquet(CLEAN_PATH)
n_rows = lf.select(pl.len()).collect(engine='streaming').item()
print(f'Jumlah baris data bersih: {n_rows:,}')

Jumlah baris data bersih: 1,497,952


## Visualisasi 1: Distribusi `synthetic_label`

Skor kesesuaian antara klaim dan artikel sumber, dari -2 sampai 2.

In [2]:
label_counts = (
    lf.group_by('synthetic_label')
    .agg(pl.len().alias('jumlah'))
    .sort('synthetic_label')
    .collect(engine='streaming')
    .with_columns(
        pl.col('synthetic_label').cast(pl.Int8).cast(pl.Utf8).alias('label'),
        (pl.col('jumlah') / n_rows * 100).round(2).alias('persen'),
    )
)

fig1 = px.bar(
    label_counts, x='label', y='jumlah', text='persen',
    title='Distribusi synthetic_label',
    labels={'label': 'Label', 'jumlah': 'Jumlah baris', 'persen': 'Persen'},
)
fig1.update_traces(texttemplate='%{text}%')
fig1.show()
label_counts

synthetic_label,jumlah,label,persen
f64,u32,str,f64
-2.0,266666,"""-2""",17.8
-1.0,111817,"""-1""",7.46
0.0,545588,"""0""",36.42
1.0,228043,"""1""",15.22
2.0,345838,"""2""",23.09


## Visualisasi 2: Distribusi `claim_potential`

In [3]:
potential_counts = (
    lf.group_by('claim_potential')
    .agg(pl.len().alias('jumlah'))
    .sort('jumlah', descending=True)
    .collect(engine='streaming')
    .with_columns((pl.col('jumlah') / n_rows * 100).round(2).alias('persen'))
)

fig2 = px.pie(
    potential_counts, names='claim_potential', values='jumlah', hole=0.4,
    title='Distribusi claim_potential',
)
fig2.show()
potential_counts

claim_potential,jumlah,persen
str,u32,f64
"""support""",870840,58.14
"""refute""",627112,41.86


## Visualisasi 3: `claim_potential` vs `synthetic_label`

Apakah kategori potensi klaim berkorelasi dengan skor akhirnya?

In [4]:
cross_tab = (
    lf.group_by(['claim_potential', 'synthetic_label'])
    .agg(pl.len().alias('jumlah'))
    .sort(['claim_potential', 'synthetic_label'])
    .collect(engine='streaming')
    .with_columns(
        pl.col('synthetic_label').cast(pl.Int8).cast(pl.Utf8).alias('label'),
        (pl.col('jumlah') / pl.col('jumlah').sum().over('claim_potential') * 100).round(1).alias('persen_dalam_kategori'),
    )
)

fig3 = px.density_heatmap(
    cross_tab, x='label', y='claim_potential', z='persen_dalam_kategori',
    histfunc='sum', text_auto=True, color_continuous_scale='Blues',
    category_orders={'label': ['-2', '-1', '0', '1', '2']},
    title='Sebaran label dalam tiap kategori claim_potential (% per kategori)',
    labels={'label': 'Label', 'claim_potential': 'Kategori potensi klaim'},
)
fig3.show()
cross_tab

claim_potential,synthetic_label,jumlah,label,persen_dalam_kategori
str,f64,u32,str,f64
"""refute""",-2.0,184332,"""-2""",29.4
"""refute""",-1.0,65186,"""-1""",10.4
"""refute""",0.0,232050,"""0""",37.0
"""refute""",1.0,83825,"""1""",13.4
"""refute""",2.0,61719,"""2""",9.8
"""support""",-2.0,82334,"""-2""",9.5
"""support""",-1.0,46631,"""-1""",5.4
"""support""",0.0,313538,"""0""",36.0
"""support""",1.0,144218,"""1""",16.6


## Visualisasi 4 dan 5: Distribusi Panjang Teks

Panjang karakter `claim` dan `source`. Data dikelompokkan (binning) di Polars lebih dulu agar grafik interaktif tetap ringan, tidak mengirim 1,5 juta titik ke browser.

In [5]:
panjang = lf.select(
    pl.col('claim').str.len_chars().alias('panjang_claim'),
    pl.col('source').str.len_chars().alias('panjang_source'),
).collect(engine='streaming')


def histogram_biner(kolom, lebar, batas_atas):
    return (
        panjang.select(((pl.col(kolom).clip(upper_bound=batas_atas) // lebar) * lebar).alias('mulai'))
        .group_by('mulai').agg(pl.len().alias('jumlah')).sort('mulai')
    )


fig4 = px.bar(
    histogram_biner('panjang_claim', 20, 600), x='mulai', y='jumlah',
    title='Distribusi panjang teks claim (karakter, bin 20; nilai >600 digabung)',
    labels={'mulai': 'Panjang claim (karakter)', 'jumlah': 'Jumlah baris'},
)
fig4.show()

fig5 = px.bar(
    histogram_biner('panjang_source', 200, 6000), x='mulai', y='jumlah',
    title='Distribusi panjang teks source (karakter, bin 200; nilai >6000 digabung)',
    labels={'mulai': 'Panjang source (karakter)', 'jumlah': 'Jumlah baris'},
)
fig5.show()

panjang.describe()

statistic,panjang_claim,panjang_source
str,f64,f64
"""count""",1.497952e6,1.497952e6
"""null_count""",0.0,0.0
"""mean""",142.794284,1609.396825
"""std""",41.17133,529.652833
"""min""",28.0,10.0
"""25%""",112.0,1275.0
"""50%""",137.0,1622.0
"""75%""",169.0,1908.0
"""max""",437.0,14418.0


## Visualisasi 6: Artikel Sumber dengan Klaim Turunan Terbanyak

Satu `claim_pmid` dapat muncul pada banyak baris karena dipasangkan dengan banyak artikel `source`. Berikut 15 artikel yang paling banyak muncul.

In [6]:
klaim_per_pmid = lf.group_by('claim_pmid').agg(pl.len().alias('jumlah_klaim'))

top_pmid = klaim_per_pmid.sort('jumlah_klaim', descending=True).head(15).collect(engine='streaming')
statistik_pmid = klaim_per_pmid.select(
    pl.len().alias('jumlah_pmid'),
    pl.col('jumlah_klaim').mean().alias('rata_rata'),
    pl.col('jumlah_klaim').median().alias('median'),
    pl.col('jumlah_klaim').max().alias('maks'),
).collect(engine='streaming')

fig6 = px.bar(
    top_pmid, x='claim_pmid', y='jumlah_klaim',
    title='Top 15 claim_pmid dengan baris terbanyak',
    labels={'claim_pmid': 'PMID artikel', 'jumlah_klaim': 'Jumlah baris'},
)
fig6.update_xaxes(type='category')
fig6.show()
statistik_pmid

jumlah_pmid,rata_rata,median,maks
u32,f64,f64,u32
426440,3.512691,3.0,10


## Visualisasi 7: Proxy Temporal Berdasarkan PMID

**Peringatan metodologis:** PMID bukan tanggal. Grafik ini hanya menunjukkan **urutan relatif** artikel (10 kelompok kuantil nilai PMID, Kelompok 1 = PMID terkecil/cenderung paling lama), bukan garis waktu kalender. Tabel konversi PMID ke tanggal tidak tersedia pada dataset ini.

In [7]:
NAMA_KELOMPOK = [f'K{i:02d}' for i in range(1, 11)]

pmid_binned = (
    lf.select(pl.col('claim_pmid').cast(pl.Int64).alias('pmid'), pl.col('synthetic_label').cast(pl.Int8).cast(pl.Utf8).alias('label'))
    .collect(engine='streaming')
    .with_columns(pl.col('pmid').qcut(10, labels=NAMA_KELOMPOK).alias('kelompok_pmid'))
)

pmid_label = (
    pmid_binned.group_by(['kelompok_pmid', 'label']).agg(pl.len().alias('jumlah_baris'))
    .sort(['kelompok_pmid', 'label'])
)
pmid_rentang = pmid_binned.group_by('kelompok_pmid').agg(
    pl.col('pmid').min().alias('pmid_min'), pl.col('pmid').max().alias('pmid_max'), pl.len().alias('jumlah_baris')
).sort('pmid_min')

fig7 = px.bar(
    pmid_label.with_columns(pl.col('kelompok_pmid').cast(pl.Utf8)),
    x='kelompok_pmid', y='jumlah_baris', color='label', barmode='stack',
    category_orders={'label': ['-2', '-1', '0', '1', '2'], 'kelompok_pmid': NAMA_KELOMPOK},
    title='Baris per kelompok PMID dan label (proxy urutan, bukan tanggal asli)',
    labels={'kelompok_pmid': 'Kelompok PMID (K01 = PMID terkecil)', 'jumlah_baris': 'Jumlah baris', 'label': 'Label'},
)
fig7.show()
pmid_rentang

kelompok_pmid,pmid_min,pmid_max,jumlah_baris
cat,i64,i64,u32
"""K01""",77,3531386,149799
"""K02""",3531428,8824240,149795
"""K03""",8824527,12100735,149797
"""K04""",12100763,17201267,149792
"""K05""",17201328,21075965,149794
"""K06""",21076015,24523447,149795
"""K07""",24523500,28097734,149794
"""K08""",28097883,31865049,149795
"""K09""",31865075,35699887,149795


## Angka Pendukung Insight

Cell ini menghitung angka dari data bersih yang dipakai pada insight di bawah.

In [8]:
print('Label dominan dalam tiap kategori claim_potential:')
dominan = (
    cross_tab.sort('persen_dalam_kategori', descending=True)
    .group_by('claim_potential', maintain_order=True).first()
)
for b in dominan.iter_rows(named=True):
    print(f"  {b['claim_potential']:<8} -> label {b['label']} ({b['persen_dalam_kategori']}%)")

print()
print('Artikel dengan baris terbanyak:', top_pmid.row(0, named=True))
print(statistik_pmid)

print()
print('Median panjang teks (karakter):')
print(panjang.select(pl.col('panjang_claim').median(), pl.col('panjang_source').median()))

Label dominan dalam tiap kategori claim_potential:
  refute   -> label 0 (37.0%)
  support  -> label 0 (36.0%)

Artikel dengan baris terbanyak: {'claim_pmid': '6451987', 'jumlah_klaim': 10}
shape: (1, 4)
┌─────────────┬───────────┬────────┬──────┐
│ jumlah_pmid ┆ rata_rata ┆ median ┆ maks │
│ ---         ┆ ---       ┆ ---    ┆ ---  │
│ u32         ┆ f64       ┆ f64    ┆ u32  │
╞═════════════╪═══════════╪════════╪══════╡
│ 426440      ┆ 3.512691  ┆ 3.0    ┆ 10   │
└─────────────┴───────────┴────────┴──────┘

Median panjang teks (karakter):
shape: (1, 2)
┌───────────────┬────────────────┐
│ panjang_claim ┆ panjang_source │
│ ---           ┆ ---            │
│ f64           ┆ f64            │
╞═══════════════╪════════════════╡
│ 137.0         ┆ 1622.0         │
└───────────────┴────────────────┘


## Ringkasan Insight

Persentase di bawah dihitung dari 1.497.981 baris pada pemeriksaan awal; pembersihan hanya membuang ±29 baris sehingga selisihnya tidak material. Angka yang bergantung pada tabel silang dan top PMID dicetak otomatis oleh cell *Angka Pendukung Insight* di atas.

1. **Distribusi label tidak seimbang** (Visualisasi 1). Label `0` (netral) paling banyak dengan 545.603 baris (36,4%), diikuti label `2` (23,1%), `-2` (17,8%), `1` (15,2%), dan `-1` paling sedikit (7,5%). Implikasi: jika label dipakai untuk melatih model klasifikasi, kelas `-1` kurang terwakili sehingga perlu class weighting atau resampling.
2. **`claim_potential` hanya dua kategori** (Visualisasi 2). `support` sebanyak 870.858 baris (58,1%) dan `refute` 627.123 baris (41,9%). Implikasi: dataset condong ke klaim yang didukung, tetapi cukup seimbang untuk klasifikasi biner tanpa penanganan khusus.
3. **`claim_potential` berhubungan dengan `synthetic_label`** (Visualisasi 3). Heatmap menampilkan persentase tiap label dalam kategori `support` dan `refute`; label dominan masing-masing kategori dicetak pada cell *Angka Pendukung Insight*. Implikasi: jika sebaran label berbeda jelas antar kategori, `claim_potential` layak menjadi fitur prediktif, tetapi juga berisiko menjadi kebocoran informasi bila dipakai untuk memprediksi label.
4. **Panjang `source` jauh lebih bervariasi daripada `claim`** (Visualisasi 4 dan 5). `claim` berkisar 28 s.d. 437 karakter dengan median 137, sedangkan `source` berkisar 10 s.d. 14.418 karakter dengan median 1.622 dan ekor kanan panjang. Implikasi: perlu strategi truncation atau padding terpisah untuk tiap kolom pada tahap NLP.
5. **Satu artikel muncul pada banyak baris** (Visualisasi 6). Terdapat 426.440 `claim_pmid` unik untuk 1.497.981 baris, rata-rata ±3,5 baris per artikel, dan sebaran di ekor atas dicetak pada cell pendukung. Implikasi: pemisahan data latih dan uji harus dilakukan per `claim_pmid` agar klaim dari artikel yang sama tidak bocor ke kedua sisi.
6. **Keterbatasan dimensi waktu** *(temuan metodologis, bukan insight dari data)*. Dataset tidak punya tanggal; Visualisasi 7 hanya menunjukkan sebaran baris dan label antar 10 kelompok kuantil PMID. Karena PMID bukan tanggal, kesimpulan soal tren waktu tidak dapat dipertanggungjawabkan.

### Keterbatasan Analisis

- **Tidak ada analisis spasial**: dataset tidak memuat informasi lokasi geografis.
- **Analisis temporal bersifat proxy**: memakai urutan kasar PMID, bukan tanggal kalender.
- **Dataset bukan dataset Indonesia**, sehingga tidak memenuhi syarat "Dataset Indonesia" pada spesifikasi tugas di README. Hal ini dicatat secara eksplisit sebagai kejujuran akademik.